# Phase 1

In [1]:
!pip install osmnx -q

In [2]:
# ============================================================
# STEP 2: Import all libraries we need for this project
# ============================================================

import osmnx as ox          # To download POI data from OpenStreetMap
import geopandas as gpd     # To work with spatial data (like pandas, but for maps)
import pandas as pd         # To work with tables and dataframes
import json                 # To read/write GeoJSON files
import warnings
warnings.filterwarnings('ignore')  # Keeps output clean

In [3]:
# ============================================================
# STEP 3: Download POI data from OpenStreetMap for Bengaluru
# ============================================================

# We define the city we want data for
city_name = "Bangalore, Karnataka, India"

# These are the POI categories we want to fetch
# OSM uses a system called "tags" — amenity is the main tag for POIs
poi_tags = {
    "amenity": [
        "hospital",       # Healthcare
        "bank",           # Financial
        "atm",            # Financial
        "restaurant",     # Food & Drink
        "fuel",           # Petrol/Gas stations
        "pharmacy",       # Healthcare
        "school",         # Education
        "police"          # Emergency services
    ]
}

print(" Downloading POI data from OpenStreetMap.")

# ox.features_from_place fetches all map features matching our tags
# inside the city boundary we named above
raw_poi = ox.features_from_place(city_name, tags=poi_tags)

print(f" Download complete!")
print(f" Total raw POIs fetched: {len(raw_poi)}")
print(f" Columns in dataset: {list(raw_poi.columns)}")

 Download complete!
 Total raw POIs fetched: 9692
 Columns in dataset: ['geometry', 'amenity', 'atm', 'branch', 'brand', 'brand:wikidata', 'brand:wikipedia', 'name', 'name:kn', 'short_name', 'addr:street', 'addr:city', 'addr:housenumber', 'addr:postcode', 'name:en', 'website', 'drive_through', 'level', 'healthcare', 'check_date', 'wheelchair', 'operator', 'operator:wikidata', 'addr:full', 'addr:state', 'email', 'phone', 'brand:en', 'brand:hi', 'brand:kn', 'brand:pa', 'brand:pnb', 'brand:ur', 'brand:wikipedia:pa', 'name:hi', 'name:pa', 'name:pnb', 'name:ur', 'old_name', 'food', 'created_by', 'opening_hours', 'shop', 'survey:date', 'addr:country', 'addr:suburb', 'capacity', 'delivery', 'official_name', 'smoking', 'start_date', 'takeaway', 'wikidata', 'cuisine', 'diet:vegetarian', 'payment:mastercard', 'payment:visa', 'source', 'addr:neighborhood', 'air_conditioning', 'payment:cash', 'payment:credit_cards', 'payment:debit_cards', 'addr:floor', 'wheelchair:description', 'alt_name', 'wikime

In [4]:
# ============================================================
# STEP 4: Inspect the raw data — understand what we received
# ============================================================

print("=== SHAPE OF DATA ===")
print(f"Rows (POIs): {raw_poi.shape[0]}")
print(f"Columns: {raw_poi.shape[1]}")

print("\n=== FIRST 3 ROWS (preview) ===")
print(raw_poi[['name', 'amenity', 'geometry']].head(3))

print("\n=== POI COUNT BY CATEGORY ===")
print(raw_poi['amenity'].value_counts())

=== SHAPE OF DATA ===
Rows (POIs): 9692
Columns: 386

=== FIRST 3 ROWS (preview) ===
                                  name     amenity                   geometry
element id                                                                   
node    247964964  State Bank of India        bank  POINT (77.60191 12.89525)
        247966855                  NaN      school   POINT (77.6026 12.87721)
        247973288         Snack Corner  restaurant  POINT (77.60107 12.89846)

=== POI COUNT BY CATEGORY ===
amenity
restaurant    3075
school        1409
bank          1405
atm           1106
hospital      1058
pharmacy       952
fuel           550
police         137
Name: count, dtype: int64


# Phase 2

In [5]:
# ============================================================
# STEP 5: Select only the columns that matter for a POI dataset
# ============================================================

# Out of 386 columns, most are empty OSM tags irrelevant to us
# A professional POI schema needs only these core fields:

columns_to_keep = [
    'name',         # POI name (e.g. "Apollo Hospital")
    'amenity',      # POI category (e.g. "hospital")
    'geometry'      # Location — the coordinates on the map
]

# Keep only those columns from our raw data
poi_clean = raw_poi[columns_to_keep].copy()

# Reset index so element/id become plain columns instead of index levels
poi_clean = poi_clean.reset_index()

print(" Columns reduced")
print(f" Shape after column selection: {poi_clean.shape}")
print(poi_clean.head(3))

 Columns reduced
 Shape after column selection: (9692, 5)
  element         id                 name     amenity  \
0    node  247964964  State Bank of India        bank   
1    node  247966855                  NaN      school   
2    node  247973288         Snack Corner  restaurant   

                    geometry  
0  POINT (77.60191 12.89525)  
1   POINT (77.6026 12.87721)  
2  POINT (77.60107 12.89846)  


In [6]:
# ============================================================
# STEP 6: Filter to POINT geometries only
# ============================================================

# OSM returns 3 types of geometry:
#   POINT     → a single coordinate (a pin on the map) ✅ we want this
#   POLYGON   → a building footprint (shape of a building) ❌ skip
#   LINESTRING→ a road or path ❌ skip
#
# TomTom POI databases work with POINT locations
# so we keep only rows where geometry is a single point

poi_clean = poi_clean[poi_clean.geometry.geom_type == 'Point'].copy()

print(f" Filtered to POINT geometries only")
print(f" POIs remaining: {len(poi_clean)}")

 Filtered to POINT geometries only
 POIs remaining: 8004


In [7]:
# ============================================================
# STEP 7: Extract lat and lon as individual columns
# ============================================================

# Right now coordinates are stored inside the 'geometry' column
# as a single POINT object like: POINT(77.60191 12.89525)
#
# We need to split that into two clean numeric columns:
# 'longitude' → the X value (East-West position)
# 'latitude'  → the Y value (North-South position)

poi_clean['longitude'] = poi_clean.geometry.x
poi_clean['latitude']  = poi_clean.geometry.y

print(" Latitude and Longitude extracted")
print(poi_clean[['name', 'amenity', 'latitude', 'longitude']].head(5))

 Latitude and Longitude extracted
                  name     amenity   latitude  longitude
0  State Bank of India        bank  12.895251  77.601909
1                  NaN      school  12.877212  77.602599
2         Snack Corner  restaurant  12.898460  77.601069
3                  NaN  restaurant  12.897312  77.601567
4           ICICI Bank        bank  12.918522  77.651842


In [8]:
# ============================================================
# STEP 8: Assign a unique POI ID to every record
# ============================================================

# In production databases, every POI must have a unique identifier
# This is how TomTom tracks, updates, and references individual POIs
# We'll create one in the format: BLR_000001, BLR_000002 ...

poi_clean['poi_id'] = [
    f"BLR_{str(i+1).zfill(6)}"   # zfill(6) pads with zeros → 000001
    for i in range(len(poi_clean))
]

print(" Unique POI IDs assigned")
print(poi_clean[['poi_id', 'name', 'amenity']].head(5))

 Unique POI IDs assigned
       poi_id                 name     amenity
0  BLR_000001  State Bank of India        bank
1  BLR_000002                  NaN      school
2  BLR_000003         Snack Corner  restaurant
3  BLR_000004                  NaN  restaurant
4  BLR_000005           ICICI Bank        bank


In [9]:
# ============================================================
# STEP 9: Arrange into final clean column order
# ============================================================

poi_clean = poi_clean[[
    'poi_id',
    'name',
    'amenity',
    'latitude',
    'longitude',
    'geometry'
]].copy()

print("=== CLEAN POI TABLE ===")
print(f"Total POIs: {len(poi_clean)}")
print(f"Columns: {list(poi_clean.columns)}")
print("\nSample rows:")
print(poi_clean.head(10).to_string())

=== CLEAN POI TABLE ===
Total POIs: 8004
Columns: ['poi_id', 'name', 'amenity', 'latitude', 'longitude', 'geometry']

Sample rows:
       poi_id                                name     amenity   latitude  longitude                   geometry
0  BLR_000001                 State Bank of India        bank  12.895251  77.601909  POINT (77.60191 12.89525)
1  BLR_000002                                 NaN      school  12.877212  77.602599   POINT (77.6026 12.87721)
2  BLR_000003                        Snack Corner  restaurant  12.898460  77.601069  POINT (77.60107 12.89846)
3  BLR_000004                                 NaN  restaurant  12.897312  77.601567  POINT (77.60157 12.89731)
4  BLR_000005                          ICICI Bank        bank  12.918522  77.651842  POINT (77.65184 12.91852)
5  BLR_000006                      Karnataka Bank        bank  12.918935  77.651830  POINT (77.65183 12.91893)
6  BLR_000007  Angel Heart Montessori Play School      school  12.918750  77.651833  POINT (

# Phase 3
The goal of this phase: Write automated rules that check every single POI and assign it a status — exactly like TomTom's data validation pipeline does before any POI goes live on their maps.


In [10]:
# ============================================================
# STEP 10: Define Bengaluru's valid coordinate boundary
# ============================================================

# Before we validate coordinates, we need to know
# what counts as "inside Bengaluru"
#
# We define a bounding box — a rectangle around the city
# Any POI whose coordinates fall OUTSIDE this box is suspect
#
# Bengaluru approximate bounding box:
#   Latitude  : 12.7 (south) to 13.2 (north)
#   Longitude : 77.3 (west)  to 77.9 (east)

BBOX = {
    'lat_min': 12.7,
    'lat_max': 13.2,
    'lon_min': 77.3,
    'lon_max': 77.9
}

print(" Bengaluru bounding box defined")
print(f"   Latitude  : {BBOX['lat_min']} → {BBOX['lat_max']}")
print(f"   Longitude : {BBOX['lon_min']} → {BBOX['lon_max']}")

 Bengaluru bounding box defined
   Latitude  : 12.7 → 13.2
   Longitude : 77.3 → 77.9


In [11]:
# ============================================================
# STEP 11: Define all validation rule functions
# ============================================================
# Each function takes ONE row of the POI table
# and returns either: 'PASS', 'FAIL', or 'REVIEW'
# along with a reason explaining why
# ============================================================

# --- RULE 1: Check if name is missing ---
def check_missing_name(row):
    """
    A POI with no name is incomplete.
    Example: A hospital with no name cannot appear on a map.
    """
    if pd.isna(row['name']) or str(row['name']).strip() == '':
        return 'FAIL', 'Missing name'
    return 'PASS', 'Name present'


# --- RULE 2: Check if coordinates are inside Bengaluru ---
def check_coordinates(row):
    """
    Coordinates must fall within Bengaluru's bounding box.
    If they don't, the POI was likely tagged in the wrong location.
    """
    lat, lon = row['latitude'], row['longitude']
    if not (BBOX['lat_min'] <= lat <= BBOX['lat_max']):
        return 'FAIL', f'Latitude {lat} outside Bengaluru range'
    if not (BBOX['lon_min'] <= lon <= BBOX['lon_max']):
        return 'FAIL', f'Longitude {lon} outside Bengaluru range'
    return 'PASS', 'Coordinates within boundary'


# --- RULE 3: Check for suspiciously short names ---
def check_name_quality(row):
    """
    Names that are only 1 character long are likely tagging errors.
    Example: A restaurant named 'A' is suspicious.
    Real POI names are at least 2 characters.
    """
    if pd.isna(row['name']):
        return 'REVIEW', 'Cannot check — name missing'
    if len(str(row['name']).strip()) == 1:
        return 'REVIEW', 'Name is only 1 character — likely a tagging error'
    return 'PASS', 'Name length acceptable'


# --- RULE 4: Check for numeric-only names ---
def check_numeric_name(row):
    """
    A POI whose name is purely a number is almost certainly wrong.
    Example: A bank named '123' is not a real bank name.
    """
    if pd.isna(row['name']):
        return 'REVIEW', 'Cannot check — name missing'
    if str(row['name']).strip().isnumeric():
        return 'FAIL', 'Name is purely numeric — invalid POI name'
    return 'PASS', 'Name is not numeric'


# --- RULE 5: Check amenity category is valid ---
def check_amenity_valid(row):
    """
    The amenity field must contain one of our known valid categories.
    Any other value means the POI was tagged with an unknown category.
    """
    valid_categories = [
        'hospital', 'bank', 'atm', 'restaurant',
        'fuel', 'pharmacy', 'school', 'police'
    ]
    if row['amenity'] not in valid_categories:
        return 'FAIL', f"Unknown category: {row['amenity']}"
    return 'PASS', 'Valid amenity category'


print(" All 5 validation rule functions defined")
print("   Rule 1 → Missing name check")
print("   Rule 2 → Coordinate boundary check")
print("   Rule 3 → Name quality check (too short)")
print("   Rule 4 → Numeric name check")
print("   Rule 5 → Valid amenity category check")

 All 5 validation rule functions defined
   Rule 1 → Missing name check
   Rule 2 → Coordinate boundary check
   Rule 3 → Name quality check (too short)
   Rule 4 → Numeric name check
   Rule 5 → Valid amenity category check


In [12]:
# ============================================================
# STEP 12: Apply all validation rules to every POI row
# ============================================================
# We loop through each rule and store results in new columns
# This gives us one status + reason column per rule
# ============================================================

# Make a working copy so our clean table stays untouched
poi_validated = poi_clean.copy()

# Apply each rule using pandas .apply()
# .apply() means: "run this function on every single row"

poi_validated[['rule1_status', 'rule1_reason']] = poi_validated.apply(
    lambda row: pd.Series(check_missing_name(row)), axis=1
)

poi_validated[['rule2_status', 'rule2_reason']] = poi_validated.apply(
    lambda row: pd.Series(check_coordinates(row)), axis=1
)

poi_validated[['rule3_status', 'rule3_reason']] = poi_validated.apply(
    lambda row: pd.Series(check_name_quality(row)), axis=1
)

poi_validated[['rule4_status', 'rule4_reason']] = poi_validated.apply(
    lambda row: pd.Series(check_numeric_name(row)), axis=1
)

poi_validated[['rule5_status', 'rule5_reason']] = poi_validated.apply(
    lambda row: pd.Series(check_amenity_valid(row)), axis=1
)

print(" All rules applied to all POIs")
print(f" Columns now: {list(poi_validated.columns)}")

 All rules applied to all POIs
 Columns now: ['poi_id', 'name', 'amenity', 'latitude', 'longitude', 'geometry', 'rule1_status', 'rule1_reason', 'rule2_status', 'rule2_reason', 'rule3_status', 'rule3_reason', 'rule4_status', 'rule4_reason', 'rule5_status', 'rule5_reason']


In [13]:
# ============================================================
# STEP 13: Assign one final overall status to every POI
# ============================================================
# Logic:
#   If ANY rule returns FAIL   → overall = FAIL
#   If ANY rule returns REVIEW → overall = REVIEW
#   If ALL rules return PASS   → overall = PASS
# ============================================================

def assign_final_status(row):
    statuses = [
        row['rule1_status'],
        row['rule2_status'],
        row['rule3_status'],
        row['rule4_status'],
        row['rule5_status']
    ]
    if 'FAIL' in statuses:
        return 'FAIL'
    elif 'REVIEW' in statuses:
        return 'REVIEW'
    else:
        return 'PASS'

poi_validated['final_status'] = poi_validated.apply(
    assign_final_status, axis=1
)

# ---- Print Summary Report ----
print("=" * 45)
print("      BENGALURU POI VALIDATION REPORT")
print("=" * 45)

total      = len(poi_validated)
passed     = (poi_validated['final_status'] == 'PASS').sum()
failed     = (poi_validated['final_status'] == 'FAIL').sum()
review     = (poi_validated['final_status'] == 'REVIEW').sum()

print(f"  Total POIs checked  : {total}")
print(f"  PASS             : {passed}  ({round(passed/total*100,1)}%)")
print(f"  FAIL             : {failed}  ({round(failed/total*100,1)}%)")
print(f"  REVIEW           : {review}  ({round(review/total*100,1)}%)")
print("=" * 45)

# Show a few failed examples
print("\n--- Sample FAIL Records ---")
print(poi_validated[poi_validated['final_status'] == 'FAIL'][
    ['poi_id','name','amenity','rule1_reason','rule2_reason','rule4_reason']
].head(5).to_string())

      BENGALURU POI VALIDATION REPORT
  Total POIs checked  : 8004
  PASS             : 7356  (91.9%)
  FAIL             : 648  (8.1%)
  REVIEW           : 0  (0.0%)

--- Sample FAIL Records ---
        poi_id name     amenity  rule1_reason                 rule2_reason                 rule4_reason
1   BLR_000002  NaN      school  Missing name  Coordinates within boundary  Cannot check — name missing
3   BLR_000004  NaN  restaurant  Missing name  Coordinates within boundary  Cannot check — name missing
19  BLR_000020  NaN    pharmacy  Missing name  Coordinates within boundary  Cannot check — name missing
30  BLR_000031  NaN    pharmacy  Missing name  Coordinates within boundary  Cannot check — name missing
31  BLR_000032  NaN    pharmacy  Missing name  Coordinates within boundary  Cannot check — name missing


In [14]:
# ============================================================
# STEP 14: Export the full validation report to CSV
# ============================================================

report_columns = [
    'poi_id', 'name', 'amenity', 'latitude', 'longitude',
    'rule1_status', 'rule1_reason',
    'rule2_status', 'rule2_reason',
    'rule3_status', 'rule3_reason',
    'rule4_status', 'rule4_reason',
    'rule5_status', 'rule5_reason',
    'final_status'
]

validation_report = poi_validated[report_columns].copy()

validation_report.to_csv('bengaluru_poi_validation_report.csv', index=False)

print(" Validation report exported!")
print(" File: bengaluru_poi_validation_report.csv")
print(f" Rows in report: {len(validation_report)}")

 Validation report exported!
 File: bengaluru_poi_validation_report.csv
 Rows in report: 8004


In [15]:
# ============================================================
# QUICK CHECK: Print your validation summary
# ============================================================

print("=" * 45)
print("      BENGALURU POI VALIDATION REPORT")
print("=" * 45)
print(f"  Total POIs checked  : {total}")
print(f"  PASS             : {passed}  ({round(passed/total*100,1)}%)")
print(f"  FAIL             : {failed}  ({round(failed/total*100,1)}%)")
print(f"  REVIEW           : {review}  ({round(review/total*100,1)}%)")
print("=" * 45)

print("\n--- Sample FAIL Records ---")
print(poi_validated[poi_validated['final_status'] == 'FAIL'][
    ['poi_id','name','amenity','rule1_reason','rule4_reason']
].head(5).to_string())

print("\n--- Sample REVIEW Records ---")
print(poi_validated[poi_validated['final_status'] == 'REVIEW'][
    ['poi_id','name','amenity','rule3_reason']
].head(5).to_string())

      BENGALURU POI VALIDATION REPORT
  Total POIs checked  : 8004
  PASS             : 7356  (91.9%)
  FAIL             : 648  (8.1%)
  REVIEW           : 0  (0.0%)

--- Sample FAIL Records ---
        poi_id name     amenity  rule1_reason                 rule4_reason
1   BLR_000002  NaN      school  Missing name  Cannot check — name missing
3   BLR_000004  NaN  restaurant  Missing name  Cannot check — name missing
19  BLR_000020  NaN    pharmacy  Missing name  Cannot check — name missing
30  BLR_000031  NaN    pharmacy  Missing name  Cannot check — name missing
31  BLR_000032  NaN    pharmacy  Missing name  Cannot check — name missing

--- Sample REVIEW Records ---
Empty DataFrame
Columns: [poi_id, name, amenity, rule3_reason]
Index: []


# Phase 4 Categorize, Enrich & Export
The goal of this phase: Take your validated dataset and transform it into a TomTom-style enriched POI schema — adding category hierarchy, priority levels, and a map-ready status flag. Then export two final deliverables: a GeoJSON and a summary CSV.

In [16]:
# ============================================================
# STEP 15: Define a TomTom-style category hierarchy
# ============================================================
# TomTom organizes POIs into:
#   main_category  → broad grouping  (e.g. "Healthcare")
#   sub_category   → specific type   (e.g. "Hospital")
#   priority_level → importance tier (1 = highest, 3 = lowest)
#
# Priority reflects how critical a POI is for navigation:
#   Level 1 → Emergency / essential services
#   Level 2 → Important daily-use services
#   Level 3 → General interest POIs
# ============================================================

category_map = {
    #  amenity tag  : (main_category,         sub_category,    priority)
    'hospital'      : ('Healthcare',           'Hospital',             1),
    'pharmacy'      : ('Healthcare',           'Pharmacy',             1),
    'police'        : ('Emergency Services',   'Police Station',       1),
    'bank'          : ('Financial Services',   'Bank',                 2),
    'atm'           : ('Financial Services',   'ATM',                  2),
    'fuel'          : ('Transport & Mobility', 'Fuel Station',         2),
    'school'        : ('Education',            'School',               2),
    'restaurant'    : ('Food & Beverage',      'Restaurant',           3),
}

print(" TomTom-style category map defined")
print(f" Categories mapped: {len(category_map)}")
print("\nCategory hierarchy preview:")
print(f"{'Amenity':<15} {'Main Category':<25} {'Sub Category':<20} {'Priority'}")
print("-" * 70)
for amenity, (main, sub, priority) in category_map.items():
    print(f"{amenity:<15} {main:<25} {sub:<20} {priority}")

 TomTom-style category map defined
 Categories mapped: 8

Category hierarchy preview:
Amenity         Main Category             Sub Category         Priority
----------------------------------------------------------------------
hospital        Healthcare                Hospital             1
pharmacy        Healthcare                Pharmacy             1
police          Emergency Services        Police Station       1
bank            Financial Services        Bank                 2
atm             Financial Services        ATM                  2
fuel            Transport & Mobility      Fuel Station         2
school          Education                 School               2
restaurant      Food & Beverage           Restaurant           3


In [17]:
# ============================================================
# STEP 16: Enrich every POI with category hierarchy columns
# ============================================================

# Work from our validated dataset
poi_enriched = poi_validated.copy()

# Apply the category map to create 3 new columns
poi_enriched['main_category'] = poi_enriched['amenity'].map(
    lambda x: category_map.get(x, ('Unknown', 'Unknown', 0))[0]
)

poi_enriched['sub_category'] = poi_enriched['amenity'].map(
    lambda x: category_map.get(x, ('Unknown', 'Unknown', 0))[1]
)

poi_enriched['priority_level'] = poi_enriched['amenity'].map(
    lambda x: category_map.get(x, ('Unknown', 'Unknown', 0))[2]
)

print(" Category enrichment applied")
print("\nSample enriched records:")
print(poi_enriched[['poi_id','name','amenity',
                     'main_category','sub_category',
                     'priority_level','final_status']].head(8).to_string())

 Category enrichment applied

Sample enriched records:
       poi_id                                name     amenity       main_category sub_category  priority_level final_status
0  BLR_000001                 State Bank of India        bank  Financial Services         Bank               2         PASS
1  BLR_000002                                 NaN      school           Education       School               2         FAIL
2  BLR_000003                        Snack Corner  restaurant     Food & Beverage   Restaurant               3         PASS
3  BLR_000004                                 NaN  restaurant     Food & Beverage   Restaurant               3         FAIL
4  BLR_000005                          ICICI Bank        bank  Financial Services         Bank               2         PASS
5  BLR_000006                      Karnataka Bank        bank  Financial Services         Bank               2         PASS
6  BLR_000007  Angel Heart Montessori Play School      school           Educa

In [18]:
# ============================================================
# STEP 17: Add a map_ready flag
# ============================================================
# map_ready = YES → POI passed validation, safe to publish
# map_ready = NO  → POI failed validation, needs fixing first
#
# This is the field TomTom's downstream systems check
# before a POI is allowed into the live map database
# ============================================================

poi_enriched['map_ready'] = poi_enriched['final_status'].map(
    lambda status: 'YES' if status == 'PASS' else 'NO'
)

# Quick summary
map_yes = (poi_enriched['map_ready'] == 'YES').sum()
map_no  = (poi_enriched['map_ready'] == 'NO').sum()

print(" Map-ready flag assigned")
print(f" Map-ready (YES) : {map_yes} POIs")
print(f" Not ready  (NO) : {map_no} POIs")

 Map-ready flag assigned
 Map-ready (YES) : 7356 POIs
 Not ready  (NO) : 648 POIs


In [19]:
# ============================================================
# STEP 18: Export map-ready POIs as GeoJSON
# ============================================================
# GeoJSON is the standard format for web maps and spatial APIs
# We export ONLY the PASS POIs — these are production-ready
# ============================================================

# Filter to map-ready only
poi_map_ready = poi_enriched[poi_enriched['map_ready'] == 'YES'].copy()

# Select final columns for GeoJSON
geojson_cols = [
    'poi_id', 'name', 'amenity',
    'main_category', 'sub_category', 'priority_level',
    'latitude', 'longitude', 'map_ready', 'geometry'
]

poi_map_ready_geo = gpd.GeoDataFrame(
    poi_map_ready[geojson_cols],
    geometry='geometry',
    crs='EPSG:4326'          # Standard WGS84 coordinate system
)

# Export to GeoJSON
poi_map_ready_geo.to_file(
    'bengaluru_poi_map_ready.geojson',
    driver='GeoJSON'
)

print(" GeoJSON exported!")
print(f" File : bengaluru_poi_map_ready.geojson")
print(f" POIs : {len(poi_map_ready_geo)}")

 GeoJSON exported!
 File : bengaluru_poi_map_ready.geojson
 POIs : 7356


In [20]:
# ============================================================
# STEP 19: Export complete enriched dataset as CSV
# ============================================================
# This includes ALL POIs (PASS + FAIL) with every column
# This is your full project deliverable — the master file
# ============================================================

final_cols = [
    'poi_id', 'name', 'amenity',
    'main_category', 'sub_category', 'priority_level',
    'latitude', 'longitude',
    'final_status', 'map_ready',
    'rule1_reason', 'rule2_reason',
    'rule3_reason', 'rule4_reason', 'rule5_reason'
]

poi_enriched[final_cols].to_csv(
    'bengaluru_poi_final.csv',
    index=False
)

print(" Final CSV exported!")
print(f" File : bengaluru_poi_final.csv")
print(f" Rows : {len(poi_enriched)}")

 Final CSV exported!
 File : bengaluru_poi_final.csv
 Rows : 8004


In [21]:
# ============================================================
# STEP 20: Print the complete project summary
# ============================================================

print("=" * 55)
print("   BENGALURU POI QUALITY PIPELINE — FINAL SUMMARY")
print("=" * 55)
print(f"  Total POIs processed       : {len(poi_enriched)}")
print(f"  Passed validation        : {map_yes} ({round(map_yes/len(poi_enriched)*100,1)}%)")
print(f"  Failed validation        : {map_no} ({round(map_no/len(poi_enriched)*100,1)}%)")
print()
print("    Output Files:")
print("     1. bengaluru_poi_validation_report.csv")
print("     2. bengaluru_poi_map_ready.geojson")
print("     3. bengaluru_poi_final.csv")
print()
print("    Map-Ready POIs by Category:")
print("-" * 55)
category_summary = poi_map_ready.groupby(
    ['main_category','sub_category','priority_level']
).size().reset_index(name='poi_count')
category_summary = category_summary.sort_values('priority_level')
print(category_summary.to_string(index=False))
print("=" * 55)

   BENGALURU POI QUALITY PIPELINE — FINAL SUMMARY
  Total POIs processed       : 8004
  Passed validation        : 7356 (91.9%)
  Failed validation        : 648 (8.1%)

    Output Files:
     1. bengaluru_poi_validation_report.csv
     2. bengaluru_poi_map_ready.geojson
     3. bengaluru_poi_final.csv

    Map-Ready POIs by Category:
-------------------------------------------------------
       main_category   sub_category  priority_level  poi_count
  Emergency Services Police Station               1         64
          Healthcare       Hospital               1        832
          Healthcare       Pharmacy               1        877
           Education         School               2        584
  Financial Services           Bank               2       1339
  Financial Services            ATM               2        630
Transport & Mobility   Fuel Station               2        124
     Food & Beverage     Restaurant               3       2906
